# Python scripts for obtaining a consensus sequence
In this notebook I will explore different ways for obtaining a consensus sequence from KEGG K01220 group. 


## Consensus sequence with Python vanilla
Here I will document my first iterations of this problem just using python withouth external third-party packages or frameworks, just to maximize my learning and refine my approach when solving problems.  
Start by importing the data just as text and save it inside a variable `contenido` and then loading my parsing function `fasta_to_dict(dna_strings:list)`, this function espects a list of strings and iterates through it and returns a dictionary where the keys are the fasta headers and the values are the sequences. 

In [13]:
with open("k01220_aa_names_aligned.fas", "r") as archivo:
    contenido = archivo.read()

def fasta_to_dict(dna_strings:list)->dict:
    seq_dict = {}
    seq=[]
    header = None
    for i in range(len(dna_strings)):
        if dna_strings[i].startswith('>'):
            if header:
                seq_dict[header] = ''.join(seq)
            header = dna_strings[i][1:]
            seq = []
        else:
            seq.append(dna_strings[i])
    if header:
        seq_dict[header] = ''.join(seq)
    return seq_dict

Tansform `contenido`, which is a string with `'\n'` characters functioning as newline, so the method `.splitlines()` uses this newline character to turn the string into a list and then can be used by `fasta_to_dict(dna_strings:list)` creating a dictionary `secuencias_dict`. 

In [15]:
secuencias_fas = contenido.splitlines()
secuencias_dict = fasta_to_dict(secuencias_fas)

Extract just the sequences from the created dictionary. The list of sequences of interest is named `lista_secuencias`. 

In [17]:
lista_secuencias = []
for i in secuencias_dict.values():
    seq = list(i)
    lista_secuencias.append(seq)
#print(lista_secuencias, sep=' ')

### Orchestrating function get_seq_consensus()
This function takes a list of sequences and a threshold as an integer (50, not 0.50) as parameters and returns a consensus sequence derived from all of them respecting the given threshold.  
Works with two variables to iterate `n,x`. The first one used to iterate through the length of a given sequence and the second one to iterate through the multiple sequences. While iterating is saving the characters in the $i,j$ column into the variable `slice_seq`which is a list and refreshes each time a iteration of a $j$ is completed. Then the orquestration begins. 

In [19]:
def get_seq_consensus(testeo_seq, threshold):
    n,x = len(testeo_seq[0]), len(testeo_seq)
    seq_w_threshold = []
    for j in range(n):
        slice_seq = []
        for i in range(x):
            slice_seq.append(testeo_seq[i][j])
            
        aa_dict = check_coincidences(slice_seq)
        aa_no_threshold = get_aa_consensus(aa_dict, x,threshold)
        seq_w_threshold.append(aa_no_threshold)
    return ''.join(seq_w_threshold)
#print(get_seq_consensus(lista_secuencias, threshold))

### Counting characters function check_coincidences()
This function takes a list `seq_slice` and generates two variables `aa_dict, n`, first one a dictionary and second one saves the number of elemnts in the list. It iterates through the list and categorizes the items in a dictionary where each key is a unique character an its value is how many times that character appears in the list. This is saved in `aa_dict` which is the final product of the function

In [21]:
#seq_slice = ['P', 'P', 'T']
def check_coincidences(seq_slice:list)->dict:
    aa_dict, n = {}, len(seq_slice)
    for i in range(n):
        aa = seq_slice[i]
        if seq_slice[i] in aa_dict:
            aa_dict[aa] += 1
        else:
            aa_dict[aa] = 1
    return aa_dict  #{'P': 2, 'T': 1}
#print(check_coincidences(seq_slice))

### Consensus character function get_aa_consensus()
This function recibes `a, x, threshold`, which are dictionary, how many sequences are we working on and the threshold of frequency. Then the function generates `a_max,empate` that are meant to keep track of the maximum values of the dictionary values and to check if there are more than one equal values. First `a_max` is initialized in 0, then we range through the dictionary (both keys and values) and if one value is bigger than `a_max` we assign that value to `a_max` and also we save the current key in `key_max`, `empate` stays the same or is changed to `False`. In this loop if we found a value that is equal to the biggest value previously found we change `empate` to `True`. However the loop must finish, because there is no guarantee that the largest values are located only at the beginning or the end. We must assume the placement is random.  
Then we check if `empate = True`, if so, that means that there are at least 2 characters that appear equal number of times. Meaning that there is a lot of variation at that position and even maybe in the sequences, and that variation should be not ignored and the function returns a gap `'-'`.   
Then we check if the `a_max` converted to frequency (little reminder: `x` is the numbers of sequences we are analyzing), with $a_max/x$, and we compare it to the threshold turned in decimal. If happens that this new frequency is bigger or equal than the threshold then, the `key_max` assciated to that value is returned by the function.  
Finally if the frequency is smaller than the threshold the function returns a gap `'-'`.  

In [23]:
#a = {'P': 2, 'T': 1, 'G':1}
#x, threshold = 6, 50
def get_aa_consensus(a:dict, x:int, threshold:int)->str:
    a_max,empate = 0, False
    for key, val in a.items():
        if val>a_max:
            a_max = val
            key_max = key
            empate = False
        elif val == a_max:
            empate = True
    if empate:
        return '-'
    elif a_max/x >= threshold/100:
        return key_max
    else: 
        return '-'
# print(get_aa_consensus(a,x,threshold))  in this case returns '-'

### Generate sequences and files
In this part we are generating 4 different files, when threshold is 50%:
- Text files:
    - consensus_threshold_50.txt
    - consensus_threshold_50_no_gaps.txt
- Fasta files:
    - consensus_threshold_50.fas
    - consensus_threshold_50_no_gaps.fas

In [24]:
new_threshold = 50
consensus_seq = get_seq_consensus(lista_secuencias, new_threshold)
print(consensus_seq)

------------------------M-K-LP-DFIFGGATAAYQAEGAT-TDGKG-VAWD--YLE----NYWYTAEPASDFY-R-YPVDL-L-E-FGVNGIRISIAWSR--IFP--G----GEVNPKGVEYYH-LFAECHKRHVEPFVTLHHFDTPE-LH--G--DFLNR-NI--FV-YA-FCF-EF--EV-YWTTFNEIGPIGDGQYLVGKFP-----------PGI-YD--KVFQSHHNMMVAHARAVKLFKD--GY-GEIGVVHALPTKY---P-DP-NPEDVRAAELEDIIHNKFILDATYLG-------YS-ETMEGV-HIL--NGG--L---DEDF---L-AAKDLNDFLGINYYMSD------------WM-AY------------D-GE-EI-HNGTG-KG-SKYQ-KGVG-R-----VPRTDWDW-IYPQ-----GLYDQI----MRVK-DYPN----YKKIYITENGLGYKDEF------D--TV-DD-RID-YVK-HLEVI-DAI-D-GANVKGYFIWSLMDVFSWSNGYEKRYGLFYVDFETQERYPKKSAYWYK--AET-------I----------------


In [17]:
with open(f"consensus_threshold_{new_threshold}.txt", "w", encoding="utf-8") as file:
    file.write(consensus_seq)

A simple function to parse the generated data into a fasta format.  

In [32]:
def text_to_fasta(text_header, separator, consensus_seq):
    return text_header+separator+consensus_seq

In [54]:
text_header = f"consensus_threshold_{new_threshold}"
separator = '\n'

consensus_seq_fasta = text_to_fasta(text_header, separator, consensus_seq)

In [58]:
with open(f"consensus_threshold_{new_threshold}.fas", "w", encoding="utf-8") as file:
    file.write(consensus_seq_fasta)

Generate sequence files without gaps because some platforms struggle with dealing with gaps on their queries

In [34]:
def no_gaps(seq):
    len_seq = len(seq)
    new_seq = ''
    for i in range(len_seq):
        if seq[i] != '-':
            new_seq = new_seq + seq[i]
    return new_seq

In [36]:
consensus_seq_no_gaps = no_gaps(consensus_seq)

text_header = f"consensus_threshold_{new_threshold}_no_gaps"
separator = '\n'
consensus_seq_no_gaps_fas = text_to_fasta(text_header, separator, consensus_seq_no_gaps)

In [38]:
with open(f"consensus_threshold_{new_threshold}_no_gaps.txt", "w", encoding="utf-8") as file:
    file.write(consensus_seq_no_gaps)

In [40]:
with open(f"consensus_threshold_{new_threshold}_no_gaps.fas", "w", encoding="utf-8") as file:
    file.write(consensus_seq_no_gaps_fas)

## Final notes
After this point only remaining code is left. I probably will come back in order to document it properly.  

In [21]:
testeo_seq = [lista_secuencias[i][20:30] for i in range(3)]

threshold = 50

for i in testeo_seq:
    print(i)
print('--------------------------------------------------')
seq_w_threshold = []


['-', '-', '-', '-', 'M', 'T', 'K', 'K', 'L', 'P']
['-', '-', '-', 'M', 'T', 'K', 'Y', 'T', 'F', 'P']
['-', '-', '-', 'M', 'P', 'K', 'Y', 'T', 'F', 'T']
--------------------------------------------------


In [22]:
#a = {'P': 2, 'T': 1, 'G':1}
#x, threshold = 6, 50
def get_aa_consensus(a:dict, x:int, threshold:int)->str:
    a_max,empate = 0, False
    for key, val in a.items():
        if val>a_max:
            a_max = val
            key_max = key
            empate = False
        elif val == a_max:
            empate = True
    if empate:
        return '-'
    elif a_max/x >= threshold/100:
        return key_max
    else: 
        return '-'
# print(get_aa_consensus(a,x,threshold)) -

#seq_slice = ['P', 'P', 'T']
def check_coincidences(seq_slice:list)->dict:
    aa_dict, n = {}, len(seq_slice)
    for i in range(n):
        aa = seq_slice[i]
        if seq_slice[i] in aa_dict:
            aa_dict[aa] += 1
        else:
            aa_dict[aa] = 1
    return aa_dict  #{'P': 2, 'T': 1}
#print(check_coincidences(seq_slice))

testeo_seq = [lista_secuencias[i][20:30] for i in range(3)]

threshold = 50

for i in testeo_seq:
    print(i)
print('--------------------------------------------------')
seq_w_threshold = []
def get_seq_consensus(testeo_seq, threshold):
    n,x = len(testeo_seq[0]), len(testeo_seq)
    for j in range(n):
        slice_seq = []
        for i in range(x):
            slice_seq.append(testeo_seq[i][j])
            
        aa_dict = check_coincidences(slice_seq)
        aa_no_threshold = get_aa_consensus(aa_dict, x,threshold)
        seq_w_threshold.append(aa_no_threshold)
    return ''.join(seq_w_threshold)
print(get_seq_consensus(lista_secuencias, threshold))

['-', '-', '-', '-', 'M', 'T', 'K', 'K', 'L', 'P']
['-', '-', '-', 'M', 'T', 'K', 'Y', 'T', 'F', 'P']
['-', '-', '-', 'M', 'P', 'K', 'Y', 'T', 'F', 'T']
--------------------------------------------------
------------------------M-K-LP-DFIFGGATAAYQAEGAT-TDGKG-VAWD--YLE----NYWYTAEPASDFY-R-YPVDL-L-E-FGVNGIRISIAWSR--IFP--G----GEVNPKGVEYYH-LFAECHKRHVEPFVTLHHFDTPE-LH--G--DFLNR-NI--FV-YA-FCF-EF--EV-YWTTFNEIGPIGDGQYLVGKFP-----------PGI-YD--KVFQSHHNMMVAHARAVKLFKD--GY-GEIGVVHALPTKY---P-DP-NPEDVRAAELEDIIHNKFILDATYLG-------YS-ETMEGV-HIL--NGG--L---DEDF---L-AAKDLNDFLGINYYMSD------------WM-AY------------D-GE-EI-HNGTG-KG-SKYQ-KGVG-R-----VPRTDWDW-IYPQ-----GLYDQI----MRVK-DYPN----YKKIYITENGLGYKDEF------D--TV-DD-RID-YVK-HLEVI-DAI-D-GANVKGYFIWSLMDVFSWSNGYEKRYGLFYVDFETQERYPKKSAYWYK--AET-------I----------------


In [23]:
from Bio import AlignIO
from collections import Counter

ModuleNotFoundError: No module named 'Bio'

In [ ]:
# Leer el alineamiento
alignment = AlignIO.read("k01220_aligned.fas", "fasta")

THRESHOLD = 0.70

consensus = []

for column in zip(*alignment):
    # Ignorar gaps
    residues = [aa for aa in column if aa != "-"]

    # Si toda la columna son gaps
    if not residues:
        consensus.append("-")
        continue

    # Aminoácido más frecuente y su frecuencia
    aminoacid, count = Counter(residues).most_common(1)[0]
    frequency = count / len(residues)

    # Aplicar umbral de conservación
    if frequency >= THRESHOLD:
        consensus.append(aminoacid)
    else:
        consensus.append("X")

consensus = "".join(consensus)

print(">consenso k01220 70%_threshold")
print(consensus)

In [ ]:
a = {'P': 2, 'T': 1, 'G':1}
x, threshold = 6, 50
def get_aa_consensus(a:dict, x:int, threshold:int)->str:
    a_max,empate = 0, False
    for key, val in a.items():
        if val>a_max:
            a_max = val
            key_max = key
            empate = False
        elif val == a_max:
            empate = True
    if empate:
        return '-'
    elif a_max/x >= threshold/100:
        return key_max
    else: 
        return '-'
print(get_aa_consensus(a,x,threshold))

In [ ]:
a = {'P': 2, 'T': 1, 'G':3}
x, threshold = 3, 40

def get_aa_consensus(a:dict, x:int, threshold:int)->str:
    a_max = max(a.values())
    aa_max = {key:round(val/x, 3)  for key, val in a.items() if val == a_max}
    
    if len(aa_max) > 1:
        return '-'
    elif next(iter(aa_max.values())) >= threshold/100:
        return next(iter(aa_max.keys()))
    else:
        return '-'
        
print(get_aa_consensus(a,x,threshold))

In [ ]:
THRESHOLD = 0.50

consensus = []

for column in zip(*alignment):
    # Ignorar gaps
    residues = [aa for aa in column if aa != "-"]

    # Si toda la columna son gaps
    if not residues:
        consensus.append("-")
        continue

    # Aminoácido más frecuente y su frecuencia
    aminoacid, count = Counter(residues).most_common(1)[0]
    frequency = count / len(residues)

    # Aplicar umbral de conservación
    if frequency >= THRESHOLD:
        consensus.append(aminoacid)
    else:
        consensus.append("X")

consensus = "".join(consensus)

print(">consenso k01220 50%_threshold")
print(consensus)